In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"

%load_ext autoreload
%autoreload 2

In [2]:
import ipdb
from Data import *
from address import *
from modelGen import *
from utils import cf_eval, cleanup_gpu
from models import CausalCondLatentCF, latentCFpp, PrototypeLatentCF
from models import CondLatentCF # Note: get the vanilla CondLatentCF
import numpy as np
import random 
from tensorflow import random as tf_random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
import argparse
import os
import pandas as pd


In [3]:
SEED = 23

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf_random.set_seed(SEED)

CF_METHODS = {  
    "LatentCFpp": latentCFpp,
    "Prototype": PrototypeLatentCF,
    "CausalCACTUS": CausalCondLatentCF,
    "CACTUS": CondLatentCF
}

In [4]:
# Experimentos que a realizar
EXP = [
    {
        "name": "LatentCF++",
        "data": "GERMANCREDIT",
        "classifier": "./exp/GERMANCREDIT_class/config.json",
        "AE": "./exp/GERMANCREDIT_AE/config.json",
        "CFmethod": "LatentCFpp",
        "context": [
            "Sex",
            "AgeAbove40"
        ],
        "epochs": 100,
        "tol": 0.001,
        "target_prob": 0.5,
        "learning_rate": 0.1
    },
    {
        "name": "Prototype C",
        "data": "GERMANCREDIT",
        "classifier": "./exp/GERMANCREDIT_class/config.json",
        "AE": "./exp/GERMANCREDIT_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "Sex",
            "AgeAbove40"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 0.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "Prototype D",
        "data": "GERMANCREDIT",
        "classifier": "./exp/GERMANCREDIT_class/config.json",
        "AE": "./exp/GERMANCREDIT_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "Sex",
            "AgeAbove40"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 100.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "CACTUS",
        "data": "GERMANCREDIT",
        "classifier": "./exp/GERMANCREDIT_class/config.json",
        "AE": "./exp/GERMANCREDIT_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "Sex",
            "AgeAbove40"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        # "dynamicAlpha": True
        "dynamicAlpha": False
    },
    {
        "name": "CausalCACTUS",
        "data": "GERMANCREDIT",
        "classifier": "./exp/GERMANCREDIT_class/config.json",
        "AE": "./exp/GERMANCREDIT_CACTUS/config.json",
        "CFmethod": "CausalCACTUS",
        "context": [
            "Sex",
            "AgeAbove40"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": True
        # "dynamicAlpha": False
    }
]


In [5]:
import re
from collections import defaultdict
# TODO: make a function to convert the causal paths for all the test samples
def build_causal_index_map(input_features, graph_str):
    """
    Returns:
        child_to_parents: dict {child_idx: [parent_idx, ...]}
        parent_to_children: dict {parent_idx: [child_idx, ...]}
        feat2idx: dict {feature_name: index}
    """

    # Feature name to index
    feat2idx = {f: i for i, f in enumerate(input_features)}

    # Extract edges using regex
    edges = re.findall(r'(\w+)\s*->\s*(\w+)', graph_str)

    child_to_parents = defaultdict(list)
    parent_to_children = defaultdict(list)

    for parent, child in edges:

        if parent not in feat2idx or child not in feat2idx:
            raise ValueError(f"Feature in graph not found in dataset: {parent} or {child}")

        p_idx = feat2idx[parent]
        c_idx = feat2idx[child]

        child_to_parents[c_idx].append(p_idx)
        parent_to_children[p_idx].append(c_idx)

    return dict(child_to_parents), dict(parent_to_children), feat2idx

# credit_rex5_constraints_min.dot
graph_str1 = """ 
strict digraph {
CreditAmount;
Duration;
Age;
CheckingAccount;
Sex;
Job;
SavingAccounts;
Housing;
Job -> Duration;
Job -> CheckingAccount;
SavingAccounts -> Job;
SavingAccounts -> CheckingAccount;
Housing -> CheckingAccount;
Housing -> Job;
}
"""

# adult_rex6_constraints_mod.dot
graph_str2 = """
strict digraph {
CreditAmount;
Duration;
Age;
CheckingAccount;
Sex;
Job;
SavingAccounts;
Housing;
SavingAccounts -> Job;
SavingAccounts -> CheckingAccount;
Housing -> CheckingAccount;
Housing -> Job;
}
"""

# # CF generation
# child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
#     data.features_lbls,
#     # graph_str
#     graph_str1
# )
# print("Feature → index:")
# print(feat2idx)
# print("\nChild → Parents (index form):")
# print(child_to_parents_dict)

In [6]:
# data.features_lbls

## Graph 1 (Minimum priors)

In [7]:
N = 50
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [12, 23, 34, 45, 56]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str1
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              

        # TODO: One-hot encoding function for datasets with catergorical features
        cat_idx = [2, 3, 4, 5] #    CheckingAccount", "Job", "Housing", "SavingAccounts"
        num_idx = [0, 1]
        X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])
        
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict, 
            cat_idx,
            num_idx
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)




********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data
Sex
1    354
0    168
Name: count, dtype: int64
AgeAbove40
0    387
1    135
Name: count, dtype: int64




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 462
Positive class: 231.0
Context 'Sex=1': 315
Context 'AgeAbove40=1': 125
----------------------------------------------------------------------------------------------------




Building model


/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_GermanCredit.py:101: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["Sex"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_GermanCredit.py:102: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["AgeAbove40"]))


Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 6)]               0         
                                                                 
 dense (Dense)               (None, 64)                448       
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                 

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


restoring weights of model AE: GERMANCREDIT_AE
Feature → index:
{'CreditAmount': 0, 'Duration': 1, 'CheckingAccount': 2, 'Job': 3, 'Housing': 4, 'SavingAccounts': 5}

Child → Parents (index form):
{1: [3], 2: [3, 5, 4], 3: [5, 4]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 model_2 (Functional)        (None, 6)                 4614      
                                                                 
 model (Functional)          (None, 2)                 35202     
                                                                 
Total params: 39,816
Trainable params: 39,048
Non-trainable params: 768
_________________________________________________________________
2/2 [==============================] - 1s 4ms/step
Gener

...

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 6)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1412        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 6)            1414        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_GermanCredit.py:101: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["Sex"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_GermanCredit.py:102: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["AgeAbove40"]))


restoring weights of model DNN: GERMANCREDIT_class
Building model
Building model: CACTUS_VAE_tabular [gen]


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 6)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1412        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 6)            1414        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

In [8]:
child_to_parents_dict

{1: [3], 2: [3, 5, 4], 3: [5, 4]}

In [9]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                        Validity          LOF  Compactness    Proximity  \
Dataset      Model                                                              
GERMANCREDIT CACTUS        0.83 ± 0.03  1.07 ± 0.01  0.39 ± 0.02  0.69 ± 0.03   
             CausalCACTUS  0.94 ± 0.03  1.06 ± 0.01  0.48 ± 0.02  0.55 ± 0.04   
             LatentCF++      1.0 ± 0.0   1.1 ± 0.01  0.57 ± 0.03  0.41 ± 0.06   
             Prototype C     1.0 ± 0.0  1.71 ± 0.09  0.11 ± 0.02   2.4 ± 0.09   
             Prototype D     1.0 ± 0.0  1.49 ± 0.03  0.14 ± 0.02  1.72 ± 0.03   

Metric                    Causal Validity (Hard) Causal Validity (Soft)  \
Dataset      Model                                                        
GERMANCREDIT CACTUS                  0.23 ± 0.08            0.71 ± 0.03   
             CausalCACTUS            0.22 ± 0.07            0.68 ± 0.04   
             LatentCF++              0.14 ± 0.04            0.69 ± 0.01   
             Prototype C             0.78 ± 0.04            0.93 ± 0.01   
             Prototype D              0.7 ± 0.08            0.89 ± 0.03   

Metric                    Causal Compact Validity inlier_fraction  
Dataset      Model                                                 
GERMANCREDIT CACTUS                   0.27 ± 0.01       1.0 ± 0.0  
             CausalCACTUS             0.32 ± 0.03       1.0 ± 0.0  
             LatentCF++               0.39 ± 0.03     0.99 ± 0.01  
             Prototype C              0.09 ± 0.02     0.55 ± 0.04  
             Prototype D              0.11 ± 0.02     0.64 ± 0.07

## Graph 2 (Moderate priors)

In [10]:
N = 50
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [11, 22, 33, 44, 55]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str2
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              

        # TODO: One-hot encoding function for datasets with catergorical features
        cat_idx = [2, 3, 4, 5] #    CheckingAccount", "Job", "Housing", "SavingAccounts"
        num_idx = [0, 1]
        X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])
        
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict, 
            cat_idx,
            num_idx
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)




********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data
Sex
1    354
0    168
Name: count, dtype: int64
AgeAbove40
0    387
1    135
Name: count, dtype: int64




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 462
Positive class: 231.0
Context 'Sex=1': 315
Context 'AgeAbove40=1': 125
----------------------------------------------------------------------------------------------------




Building model
Building model: DNN [class]
Model: "model"
_________________________________________________________________

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_GermanCredit.py:101: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["Sex"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_GermanCredit.py:102: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["AgeAbove40"]))


Non-trainable params: 768
_________________________________________________________________
restoring weights of model DNN: GERMANCREDIT_class
Building model
Building model: AE [gen]
Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 6)]               0         
                                                                 
 model_1 (Functional)        (None, 16)                4624      
                                                                 
 model_2 (Functional)        (None, 6)                 4614      
                                                                 
Total params: 9,238
Trainable params: 9,238
Non-trainable params: 0
_________________________________________________________________
restoring weights of model AE: GERMANCREDIT_AE


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Feature → index:
{'CreditAmount': 0, 'Duration': 1, 'CheckingAccount': 2, 'Job': 3, 'Housing': 4, 'SavingAccounts': 5}

Child → Parents (index form):
{3: [5, 4], 2: [5, 4]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 model_2 (Functional)        (None, 6)                 4614      
                                                                 
 model (Functional)          (None, 2)                 35202     
                                                                 
Total params: 39,816
Trainable params: 39,048
Non-trainable params: 768
_________________________________________________________________
2/2 [==============================] - 0s 3ms/step
Generating CF 0/50
Generating CF 1/50
Generating CF 2/50
Genera

...

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 6)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1412        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 6)            1414        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_GermanCredit.py:101: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["Sex"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_GermanCredit.py:102: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["AgeAbove40"]))


Total params: 35,202
Trainable params: 34,434
Non-trainable params: 768
_________________________________________________________________
restoring weights of model DNN: GERMANCREDIT_class
Building model
Building model: CACTUS_VAE_tabular [gen]


/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 6)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1412        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 6)            1414        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

In [11]:
child_to_parents_dict

{3: [5, 4], 2: [5, 4]}

In [12]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                        Validity          LOF  Compactness    Proximity  \
Dataset      Model                                                              
GERMANCREDIT CACTUS        0.84 ± 0.05  1.09 ± 0.01  0.39 ± 0.02  0.66 ± 0.02   
             CausalCACTUS  0.98 ± 0.03  1.06 ± 0.02  0.45 ± 0.04  0.57 ± 0.03   
             LatentCF++      1.0 ± 0.0  1.09 ± 0.02  0.58 ± 0.02  0.37 ± 0.04   
             Prototype C     1.0 ± 0.0  1.69 ± 0.07   0.1 ± 0.01  2.36 ± 0.03   
             Prototype D     1.0 ± 0.0  1.51 ± 0.02  0.16 ± 0.02  1.67 ± 0.04   

Metric                    Causal Validity (Hard) Causal Validity (Soft)  \
Dataset      Model                                                        
GERMANCREDIT CACTUS                  0.82 ± 0.04            0.89 ± 0.03   
             CausalCACTUS            0.77 ± 0.05            0.87 ± 0.04   
             LatentCF++              0.94 ± 0.01            0.97 ± 0.01   
             Prototype C             0.99 ± 0.01            0.99 ± 0.01   
             Prototype D             0.97 ± 0.02            0.98 ± 0.01   

Metric                    Causal Compact Validity inlier_fraction  
Dataset      Model                                                 
GERMANCREDIT CACTUS                   0.35 ± 0.02       1.0 ± 0.0  
             CausalCACTUS              0.4 ± 0.04       1.0 ± 0.0  
             LatentCF++               0.57 ± 0.02       1.0 ± 0.0  
             Prototype C               0.1 ± 0.01      0.6 ± 0.03  
             Prototype D              0.16 ± 0.02     0.63 ± 0.06

In [13]:
# TODO: One-hot encoding function for datasets with catergorical features
cat_idx = [2, 3, 4, 5] #    CheckingAccount", "Job", "Housing", "SavingAccounts"
num_idx = [0, 1]
X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])


# --- CF evaluation ---
cf_scores, cf_scores_labels = cf_eval(
    data.X_train, # previous name: data.X_train
    context_training, # previous name: context_training,
    X_test, # previous name: x,
    X_cf, # previous name: cf_x, 
    y_original_labels, # previous name: y_,
    context_test, # previous name: context,
    y_cf_labels, # previous name: cf_y_
    data.scaler_inverse_transform(X_test),
    data.scaler_inverse_transform(X_cf),
    child_to_parents_dict,
    cat_idx,
    num_idx
)

print(cf_scores, cf_scores_labels)

[0 0 1 1 1 1 0 0 0 0 1 1 1 1 0 0 1 1 1 1 1 1 0 0 0 0 1 0 0 1 0 1 1 0 0 0 0
 0 0 0 1 0 0 0 1 0 1 0 0 1]
[0 0 1 1 1 1 0 0 0 0 1 1 1 1 0 0 1 0 1 1 1 1 0 0 0 0 0 0 0 1 0 1 1 0 0 0 0
 0 0 0 1 0 0 0 1 0 1 0 0 1]
0.96
0.9721729666029966 1.037178187546966 1.3707737209591775 50
[1.4037098890891124, 0.5730621625278489, 0.96, 0.43, 1.0564613794714903, 0.82, 0.9, 0.38666666666666666, 1.0] ['proximity', 'n_proximity', 'validity', 'compactness', 'lof_context', 'causal_validity_hard', 'causal_validity_soft', 'causal_compact_val', 'inlier_fraction']
